In [9]:
import numpy as np
import pandas as pd
import json
from google.colab import files


def frequencia_sintonia(L_mH, C_uF):
    """
    Calcula a frequência de sintonia de um filtro LC.

    Parâmetros:
    L (float): Indutância em henries (H).
    C (float): Capacitância em farads (F).

    Retorna:
    float: Frequência de sintonia em hertz (Hz).
    """
    L = L_mH * 1e-3  # Convertendo para H
    C = C_uF * 1e-6  # Convertendo para F
    if L <= 0 or C <= 0:
        raise ValueError("A indutância (L) e a capacitância (C) devem ser maiores que zero.")

    f = 1 / (2 * np.pi * np.sqrt(L * C))
    return f

# Funções de cálculo
def calcular_impedancia_amortecido(R, r, L, C,w):
    """
    Calcula a impedância de cada elemento (R, L e C).
    """
    Z_R = complex(R, 0)                     # Impedância do resistor
    Z_L = complex(r, w * L * 1e-3)          # Impedância do indutor (L em mH)
    Z_C = complex(0, -1 / (w * C * 1e-6))   # Impedância do capacitor (C em uF)
    Z_RL = 1 / ( 1/Z_R + 1/Z_L )            # Impedância em paralela entre R e L
    Z_F = Z_RL + Z_C                    # Impedância total
    return Z_R, Z_L, Z_C, Z_RL, Z_F

def calcular_corrente_tensao(V_linha, Z_R, Z_L, Z_C, Z_RL, Z_F):
    """
    Calcula a corrente em cada elemento a partir da tensão de linha.
    """
    V_fase = V_linha / np.sqrt(3)

    I_F = V_fase / Z_F

    V_R = I_F * Z_RL
    V_L = I_F * Z_RL
    V_C = I_F * Z_C
    V_F = V_R + V_C

    I_R = V_R / Z_R
    I_L = V_L / Z_L
    I_C = V_C / Z_C


    return I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F


def calcular_potencia(I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F):
    """
    Calcula a potência em cada elemento. Considera a tensão de fase e a corrente em cada componente.
    """
    P_R = V_R * np.conj(I_R)
    P_L = V_L * np.conj(I_L)
    P_C = V_C * np.conj(I_C)
    P_F = V_F * np.conj(I_F)

    return P_R, P_L, P_C, P_F

# Função principal para cálculo dos parâmetros do filtro
def calcular_filtro_estrela(R, r, L, C, V_linha, w):
    """
    Função principal para calcular impedâncias, correntes, tensões e potências no filtro em estrela.
    """
    # Cálculo das impedâncias
    Z_R, Z_L, Z_C, Z_RL, Z_F = calcular_impedancia_amortecido(R, r, L, C, w)

    # Cálculo da corrente em cada elemento
    I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F = calcular_corrente_tensao(V_linha, Z_R, Z_L, Z_C, Z_RL, Z_F)

    # Cálculo da potência em cada elemento
    P_R, P_L, P_C, P_F = calcular_potencia(I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F)

    # Retornando o dicionário e os valores individuais
    return (
        Z_R, Z_L, Z_C, Z_F,
        I_R, I_L, I_C, I_F,
        V_R, V_L, V_C, V_F,
        P_R, P_L, P_C, P_F
    )

def celulas(nr_cap_serie, nr_cap_paral, V_C, P_C, sobretensao_capacitores):
    qt_cap = 3 * nr_cap_serie * nr_cap_paral
    tensao_nominal_celula = np.abs(V_C) * sobretensao_capacitores / nr_cap_serie
    potencia_nominal_celula = 3 * np.abs(P_C) * sobretensao_capacitores**2 / qt_cap
    capacitancia_nominal_celula = 1 / ( w1 * tensao_nominal_celula **2 / potencia_nominal_celula )
    capacitancia_associacao = capacitancia_nominal_celula * nr_cap_paral / nr_cap_serie

    return qt_cap, tensao_nominal_celula, potencia_nominal_celula, capacitancia_nominal_celula, capacitancia_associacao


# Exibir o dicionário resultado_formatado de forma organizada
def exibir_resultados(resultados):
    for categoria, valores in resultados.items():
        print(f"=== {categoria} ===")
        for elemento, valor in valores.items():
            print(f"{elemento}: {valor}")
        print("\n")

def short_circuit_current(line_voltage, inductance_mH, frequency):
    """
    Calculate the short-circuit current of an inductor.

    Parameters:
    voltage (float): Voltage in volts.
    inductance (float): Inductance in henries.
    frequency (float): Frequency in hertz.

    Returns:
    float: Short-circuit current in amperes.
    """
    inductance = inductance_mH * 1e-3  # Convert to henry
    line_voltage = line_voltage * 1e3  # Convert to volt
    # Calculate the inductive reactance
    X_L = 2 * np.pi * frequency * inductance

    # Calculate the short-circuit current
    current = ( line_voltage/np.sqrt(3) ) / X_L

    return current



def exportar_resultados(dicionario_resultados, nome_base="resultados_filtro_harmonico_amortecido"):
    """
    Exporta o dicionário para arquivos Excel, Texto e JSON, e faz o download automaticamente no Google Colab.

    Parâmetros:
    - dicionario_resultados (dict): Dicionário com os dados organizados em seções.
    - nome_base (str): Base do nome dos arquivos gerados (padrão: "resultados_filtro_harmonico").
    """
    # Exportando para Excel
    nome_excel = f"{nome_base}.xlsx"
    excel_data = {}
    for sheet_name, data in dicionario_resultados.items():
        df = pd.DataFrame(data.items(), columns=["Parâmetro", "Valor"])
        excel_data[sheet_name] = df

    with pd.ExcelWriter(nome_excel) as writer:
        for sheet_name, df in excel_data.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    # Exportando para Texto
    nome_texto = f"{nome_base}.txt"
    with open(nome_texto, "w") as file:
        for section, data in dicionario_resultados.items():
            file.write(f"==== {section} ====\n")
            for parametro, valor in data.items():
                file.write(f"{parametro}: {valor}\n")
            file.write("\n")  # Linha em branco para separar as seções

    # Exportando para JSON
    nome_json = f"{nome_base}.json"
    with open(nome_json, "w") as file:
        json.dump(dicionario_resultados, file, indent=4)

    # Fazendo download dos arquivos no Google Colab
    files.download(nome_excel)
    files.download(nome_texto)
    files.download(nome_json)


In [10]:
import numpy as np

def formatar_resultados(Z_R, Z_L, Z_C, Z_F, I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F, P_R, P_L, P_C, P_F,
                        qt_cap, nr_cap_serie, nr_cap_paral, tensao_nominal_celula, potencia_nominal_celula,
                        capacitancia_nominal_celula, sobretensao_capacitores, sobrecorrente_indutor,
                        corrente_curto_indutor, L, r, sobrecorrente_resistor, V_linha, R):

    resultados_formatados = {
        "Impedancia": {
            "R (ohm)": f"{np.abs(Z_R):.2f} ∠ {np.angle(Z_R, deg=True):.2f}°",
            "L (ohm)": f"{np.abs(Z_L):.2f} ∠ {np.angle(Z_L, deg=True):.2f}°",
            "C (ohm)": f"{np.abs(Z_C):.2f} ∠ {np.angle(Z_C, deg=True):.2f}°",
            "Filtro (ohm)": f"{np.abs(Z_F):.2f} ∠ {np.angle(Z_F, deg=True):.2f}°",
            "Sintonia (Hz)": f"{frequencia_sintonia(L, C):.2f}"
        },
        "Corrente": {
            "R (A)": f"{np.abs(I_R):.2f} ∠ {np.angle(I_R, deg=True):.2f}°",
            "L (A)": f"{np.abs(I_L):.2f} ∠ {np.angle(I_L, deg=True):.2f}°",
            "C (A)": f"{np.abs(I_C):.2f} ∠ {np.angle(I_C, deg=True):.2f}°",
            "Filtro (A)": f"{np.abs(I_F):.2f} ∠ {np.angle(I_F, deg=True):.2f}°"
        },
        "Tensao": {
            "R (V)": f"{np.abs(V_R):.2f} ∠ {np.angle(V_R, deg=True):.2f}°",
            "L (V)": f"{np.abs(V_L):.2f} ∠ {np.angle(V_L, deg=True):.2f}°",
            "C (V)": f"{np.abs(V_C):.2f} ∠ {np.angle(V_C, deg=True):.2f}°",
            "Filtro (V)": f"{np.abs(V_F):.2f} ∠ {np.angle(V_F, deg=True):.2f}°"
        },
        "Potencia": {
            "R (MW)": f"{3 * np.real(P_R) / 1e6:.2f}",
            "L (MW + jMVAr)": f"{3 * P_L / 1e6:.2f}",
            "C (MVAr)": f"{3 * np.imag(P_C) / 1e6:.2f}",
            "Filtro (MW + jMVAr)": f"{3 * P_F / 1e6:.2f}"
        },
        "Celulas Capacitivas": {
            "Quantidade total de células": f"{qt_cap:.0f}",
            "Quantidade série de células": f"{nr_cap_serie:.0f}",
            "Quantidade paralela de células": f"{nr_cap_paral:.0f}",
            "Tensão da célula (kV)": f"{tensao_nominal_celula / 1e3:.4f}",
            "Potência da célula (kVAr)": f"{potencia_nominal_celula / 1e3:.4f}",
            "Capacitância da célula (uF)": f"{capacitancia_nominal_celula * 1e6:.4f}"
        },
        "Banco": {
            "Tensão do banco (kV)": f"{np.sqrt(3) * sobretensao_capacitores * np.abs(V_C) / 1000:.4f}",
            "Sobretensão Considerada": f"{sobretensao_capacitores:.4f}",
            "Potência do banco (MVAr)": f"{qt_cap * potencia_nominal_celula / 1e6:.4f}",
            "Capacitância do banco (uF)": f"{capacitancia_nominal_celula * nr_cap_paral / nr_cap_serie * 1e6:.4f}"
        },
        "Indutor": {
            "Corrente Nominal (A)": f"{sobrecorrente_indutor * np.abs(I_L):.4f}",
            "Sobrecorrente Considerada": f"{sobrecorrente_indutor:.4f}",
            "Corrente de curto-circuito (kA)": f"{corrente_curto_indutor / 1e3:.4f}",
            "Indutancia (mH)": f"{L:.4f}",
            "resistencia do indutor (ohms)": f"{r:.4f}",
        },
        "Resistor": {
            "Corrente Nominal (A)": f"{sobrecorrente_resistor * np.abs(I_R):.4f}",
            "Sobrecorrente Considerada": f"{sobrecorrente_resistor:.4f}",
            "Corrente de curto-circuito (kA)": f"{(V_linha/np.sqrt(3)) / R:.4f}",
            "Resistencia (ohms)": f"{R:.4f}",
        }
    }

    return resultados_formatados


In [11]:
#############################################################################################
# Os dados de entrada iniciam aqui
f1 = 60
w1 = 2 * np.pi * f1
R = 222      # Resistencia em ohms
r = 0.792        # resistência do indutor -->0
L = 34.303     # Indutancia em mH
C = 8.543       # Capacitancia em uF
V_linha = 34.5  # Tensao de linha em kV
sobretensao_capacitores = 1.3
sobrecorrente_indutor = 1.66
sobrecorrente_resistor = 1.66
nr_cap_serie = 2
nr_cap_paral = 2
# Os dados de entrada terminam aqui
#############################################################################################


Z_R, Z_L, Z_C, Z_F, I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F, P_R, P_L, P_C, P_F = calcular_filtro_estrela(R, r, L, C, V_linha * 1e3, w1)
qt_cap, tensao_nominal_celula, potencia_nominal_celula, capacitancia_nominal_celula, capacitancia_associacao = celulas(nr_cap_serie, nr_cap_paral, V_C, P_C, sobretensao_capacitores)
corrente_curto_indutor = short_circuit_current(V_linha, L, f1)

resultados = formatar_resultados(Z_R, Z_L, Z_C, Z_F, I_R, I_L, I_C, I_F, V_R, V_L, V_C, V_F,
                                 P_R, P_L, P_C, P_F, qt_cap, nr_cap_serie, nr_cap_paral,
                                 tensao_nominal_celula, potencia_nominal_celula,
                                 capacitancia_nominal_celula, sobretensao_capacitores,
                                 sobrecorrente_indutor, corrente_curto_indutor, L, r,
                                 sobrecorrente_resistor, V_linha, R)


exportar_resultados(resultados)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
exibir_resultados(resultados)

=== Impedancia ===
R (ohm): 222.00 ∠ 0.00°
L (ohm): 12.96 ∠ 86.50°
C (ohm): 310.50 ∠ -90.00°
Filtro (ohm): 297.70 ∠ -89.71°
Sintonia (Hz): 294.00


=== Corrente ===
R (A): 3.88 ∠ 172.88°
L (A): 66.56 ∠ 86.38°
C (A): 66.91 ∠ 89.71°
Filtro (A): 66.91 ∠ 89.71°


=== Tensao ===
R (V): 862.33 ∠ 172.88°
L (V): 862.33 ∠ 172.88°
C (V): 20774.53 ∠ -0.29°
Filtro (V): 19918.58 ∠ 0.00°


=== Potencia ===
R (MW): 0.01
L (MW + jMVAr): 0.01+0.17j
C (MVAr): -4.17
Filtro (MW + jMVAr): 0.02-4.00j


=== Celulas Capacitivas ===
Quantidade total de células: 12
Quantidade série de células: 2
Quantidade paralela de células: 2
Tensão da célula (kV): 13.5034
Potência da célula (kVAr): 587.2605
Capacitância da célula (uF): 8.5430


=== Banco ===
Tensão do banco (kV): 46.7773
Sobretensão Considerada: 1.3000
Potência do banco (MVAr): 7.0471
Capacitância do banco (uF): 8.5430


=== Indutor ===
Corrente Nominal (A): 110.4851
Sobrecorrente Considerada: 1.6600
Corrente de curto-circuito (kA): 1.5403
Indutancia (mH): 